# 05 - Gradient boosting

Does a non-linear model beat the linear one from notebook 04? Same setup: train on
2022-2023, score on 2024, target `delta_s`, MAE as the headline metric.

Model settings are chosen by cross-validation **inside the training seasons only**
(folds grouped by race, so laps from one race never sit on both sides of a split).
2024 is not used to pick anything.

In [1]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from f1lab.features import prepare, split, mae, rmse
from f1lab.models import (GBM_FEATURES, gbm_features, linear_features, make_gbm, make_linear)

raw = pd.read_parquet("../data/laps.parquet")
clean = pd.read_parquet("../data/laps_clean.parquet")
train, val = split(prepare(raw, clean), "dev")
tracks = sorted(train["track"].unique())
X, y = gbm_features(train, tracks), train["delta_s"]
groups = train["season"].astype(str) + "-" + train["round"].astype(str)

## Choosing model size by grouped cross-validation (training seasons only)

In [2]:
rows = []
for leaves, trees in itertools.product([3, 4, 6, 8, 16, 31], [60, 100, 150, 200, 500]):
    scores = []
    for fit_idx, test_idx in GroupKFold(5).split(X, y, groups):
        m = make_gbm(leaves, trees).fit(X.iloc[fit_idx], y.iloc[fit_idx])
        scores.append(mae(y.iloc[test_idx], pd.Series(m.predict(X.iloc[test_idx]), index=y.index[test_idx])))
    rows.append((leaves, trees, np.mean(scores)))
cv = pd.DataFrame(rows, columns=["leaves", "trees", "cv MAE (s)"]).sort_values("cv MAE (s)").reset_index(drop=True)
best_leaves, best_trees = int(cv.loc[0, "leaves"]), int(cv.loc[0, "trees"])
print(f"chosen: {best_leaves} leaves, {best_trees} trees")
cv.head(5).round(4)

chosen: 4 leaves, 200 trees


,leaves,trees,cv MAE (s)
0,4,200,0.6098
1,3,200,0.6107
2,4,150,0.6109
3,6,150,0.6110
4,16,60,0.6115


In [3]:
# how does CV error move with model size?
cv.pivot(index="leaves", columns="trees", values="cv MAE (s)").round(3)

trees,60,100,150,200,500
leaves,,,,,
3,0.632,0.623,0.614,0.611,0.617
4,0.626,0.615,0.611,0.610,0.618
6,0.620,0.613,0.611,0.613,0.623
8,0.617,0.614,0.616,0.618,0.626
16,0.611,0.613,0.617,0.621,0.633
31,0.613,0.618,0.622,0.628,0.640


Bigger is not better. CV error is lowest for small models (3 to 6 leaves, around 150
to 200 trees, about 0.61) and rises steadily as the model grows, reaching 0.640 for
31 leaves and 500 trees. With only two seasons of training data, big trees fit
race-specific noise. The chosen model is deliberately small.

## Validation on 2024, against everything so far

In [4]:
def gbm_val(columns=None, seeds=range(5)):
    cols = columns or GBM_FEATURES
    scores = []
    for seed in seeds:
        m = make_gbm(best_leaves, best_trees, seed).fit(gbm_features(train, tracks, cols), y)
        pred = pd.Series(m.predict(gbm_features(val, tracks, cols)), index=val.index)
        scores.append((mae(val["delta_s"], pred), rmse(val["delta_s"], pred)))
    return np.array(scores), m, pred

scores, gbm, gbm_pred = gbm_val()
lin = make_linear().fit(linear_features(train), y)
lin_pred = pd.Series(lin.predict(linear_features(val)), index=val.index)
summary = pd.DataFrame(
    {
        "MAE (s)": {
            "best baseline (compound + tyre-age bucket)": 0.882,
            "linear, Huber (notebook 04)": mae(val["delta_s"], lin_pred),
            "gradient boosting (mean of 5 seeds)": scores[:, 0].mean(),
        }
    }
).round(3)
print(f"gradient boosting MAE by seed: {scores[:, 0].round(3)} (spread {scores[:, 0].std():.4f})")
summary

gradient boosting MAE by seed: [0.65  0.649 0.655 0.649 0.651] (spread 0.0021)


,MAE (s)
best baseline (compound + tyre-age bucket),0.882
"linear, Huber (notebook 04)",0.662
gradient boosting (mean of 5 seeds),0.651


Gradient boosting reaches 0.651 s, against 0.662 for the linear model and 0.882 for the
best baseline. The five seeds agree to within 0.002, so the 0.011 s edge over the
linear model is real, but it is small: about 2%. Nearly all of the improvement over the
baselines came from adding the fuel term in notebook 04; a non-linear model adds little
on top of it.

## Which inputs matter? (ablations, 5 seeds each)

In [5]:
def without(*drop):
    return [c for c in GBM_FEATURES if c not in drop]

variants = {
    "all features": GBM_FEATURES,
    "no track": without("track"),
    "no temperatures": without("track_temp", "air_temp"),
    "no compound": without("compound"),
    "no tyre age": without("tyre_age"),
    "no fuel features": without("fuel_frac", "laps_remaining", "lap_number"),
}
ablation = pd.Series({name: gbm_val(cols)[0][:, 0].mean() for name, cols in variants.items()}, name="MAE (s)")
pd.DataFrame({"MAE (s)": ablation, "change vs all features": ablation - ablation["all features"]}).round(3)

,MAE (s),change vs all features
all features,0.651,0.000
no track,0.645,-0.006
no temperatures,0.660,0.009
no compound,0.652,0.001
no tyre age,0.693,0.042
no fuel features,0.878,0.227


The ablation above was run on 2024, so using it to decide which inputs to keep would
make 2024 a slightly optimistic score. A feature decision should be made on the
training seasons. The same comparison by grouped cross-validation, without touching 2024:

In [6]:
def cv_mae(columns):
    Xc = gbm_features(train, tracks, columns)
    scores = []
    for seed in range(3):
        for fit_idx, test_idx in GroupKFold(5).split(Xc, y, groups):
            m = make_gbm(best_leaves, best_trees, seed).fit(Xc.iloc[fit_idx], y.iloc[fit_idx])
            scores.append(mae(y.iloc[test_idx], pd.Series(m.predict(Xc.iloc[test_idx]), index=y.index[test_idx])))
    return np.mean(scores)

cv_ablation = pd.Series({name: cv_mae(cols) for name, cols in variants.items()}, name="CV MAE (s)")
pd.DataFrame({"CV MAE (s)": cv_ablation, "change vs all features": cv_ablation - cv_ablation["all features"]}).round(3)

,CV MAE (s),change vs all features
all features,0.610,0.000
no track,0.610,-0.000
no temperatures,0.610,0.000
no compound,0.611,0.001
no tyre age,0.650,0.040
no fuel features,0.870,0.260


By cross-validation on the training seasons, the track, temperatures and compound
make no measurable difference (within 0.001 s). Only two inputs clearly matter: **tyre
age** (+0.040 s when removed) and **the fuel features** (+0.260 s). The 2024 ablation
above suggested dropping the track would help (-0.006) and temperatures would hurt
(+0.009), but cross-validation does not support either, so those are 2024-specific
and I am keeping the full feature set rather than tuning to one season.

In [7]:
imp = pd.Series(gbm.booster_.feature_importance("gain"), index=GBM_FEATURES)
(imp / imp.sum()).sort_values(ascending=False).round(3)

fuel_frac         0.442
lap_number        0.212
track             0.164
tyre_age          0.114
laps_remaining    0.045
track_temp        0.014
air_temp          0.007
race_laps         0.001
compound          0.001
dtype: float64

Gain importance ranks the track third (0.164). Yet removing it changes nothing in
cross-validation. The model *uses* the track, but that does not translate into
better predictions on unseen races. Importance measures use, not benefit.

## Where the errors are

In [8]:
err = (val["delta_s"] - gbm_pred).abs()
stage = pd.cut(1 - val["laps_remaining"] / val["race_laps"], [0, .25, .5, .75, 1])
lin_err = (val["delta_s"] - lin_pred).abs()
pd.DataFrame({"gradient boosting": err.groupby(stage, observed=True).mean(), "linear": lin_err.groupby(stage, observed=True).mean()}).round(3)

,gradient boosting,linear
"(0.0, 0.25]",0.693,0.657
"(0.25, 0.5]",0.586,0.542
"(0.5, 0.75]",0.557,0.601
"(0.75, 1.0]",0.758,0.828


The two models fail in different places. Gradient boosting is better late in the race
(0.758 against 0.828 in the final quarter, 0.557 against 0.601 in the third) but worse
early (0.693 against 0.657, 0.586 against 0.542). I have not tested why.

In [9]:
unseen = val["track"] == "Shanghai"
print(f"Shanghai (unseen): GBM {err[unseen].mean():.3f}, linear {lin_err[unseen].mean():.3f}")
print(f"everywhere else:   GBM {err[~unseen].mean():.3f}, linear {lin_err[~unseen].mean():.3f}")

Shanghai (unseen): GBM 0.885, linear 0.898
everywhere else:   GBM 0.643, linear 0.653


Both models are poor at Shanghai, which has no training data (0.885 and 0.898), but
equally poor, so a non-linear model does not rescue an unseen track. Elsewhere gradient
boosting leads by 0.01 s (0.643 against 0.653).

## Takeaways
- The big win is fuel. Everything else is small.
- Tyre age is the only other input that clearly helps. Compound, temperatures and
  track show no robust benefit from two training seasons.
- A non-linear model beats the linear one by about 2%. That is worth reporting but
  not worth a more complicated system on its own.
- Remaining error of about 0.65 s looks like mostly noise the inputs cannot explain
  (traffic, incidents). Point predictions may be near their ceiling with this data, so
  the more useful output is a calibrated *range*, which is where the project goes next.